# Prepare Bundestag 2021 constituency vote data

This notebook runs the new local-file preparation pipeline. It does not download anything and does not reimplement the calculation logic. The official polling-district CSV is aggregated to constituencies before `first_votes.json` and `second_votes.json` are written.

The representative statistics use two published gender categories. Their source note states that the male category also includes people recorded as diverse and people without a gender entry in the birth register.

In [ ]:
from pathlib import Path
import sys


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository checkout")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
ROOT

## Local input files

Adjust these paths to the downloaded CSV files. The third file is optional. When present, it supplies the federal party × gender × age × postal/in-person pattern used as the IPF seed. When absent, the pipeline uses the same exact margins with an independence seed.

In [ ]:
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw21_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw21_rws_stimmabgabe_laender.csv"
FEDERAL_METHOD_DEMOGRAPHICS_CSV = None  # e.g. ROOT / "scripts/data/btw21_rws_stimmabgabe_bezirksart.csv"
OUTPUT_DIRECTORY = ROOT / "scripts/data/generated"

## Run the single preparation process

The Python pipeline performs all parsing, party fallback selection, iterative proportional fitting, constituency expansion, validation and JSON export.

In [ ]:
from scripts.election_data import prepare_btw2021_vote_entries

result = prepare_btw2021_vote_entries(
    district_results_csv=DISTRICT_RESULTS_CSV,
    state_demographics_csv=STATE_DEMOGRAPHICS_CSV,
    federal_method_demographics_csv=FEDERAL_METHOD_DEMOGRAPHICS_CSV,
    output_directory=OUTPUT_DIRECTORY,
)
result.validation

In [ ]:
{
    "first_votes": len(result.firstVotes),
    "second_votes": len(result.secondVotes),
    "first_votes_path": result.firstVotesPath,
    "second_votes_path": result.secondVotesPath,
    "districts": int(result.districtTotals["districtId"].nunique()),
    "parties": int(result.districtTotals["party"].nunique()),
}

## Inspect modelling choices

Parties that have their own representative-statistics profile use it. Other parties, including the SSW when no separate profile exists, use `Sonstige`; a uniform profile is the final fallback.

In [ ]:
result.profiles[
    ["party", "demographicProfileSource", "methodSeedSource"]
].drop_duplicates().sort_values(["demographicProfileSource", "party"])

In [ ]:
result.profiles.head(24)